# Heat Equation with PINNs

This notebook demonstrates how to solve the **1D heat equation** using Physics-Informed Neural Networks (PINNs).

## The Problem

We solve the heat diffusion equation:

$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$$

with:
- **Domain**: $x \in [0, 1]$, $t \in [0, 1]$
- **Boundary conditions**: $u(0, t) = u(1, t) = 0$
- **Initial condition**: $u(x, 0) = \sin(\pi x)$

## Analytical Solution

The analytical solution is:

$$u(x, t) = e^{-\alpha \pi^2 t} \sin(\pi x)$$

We will compare our PINN solution against this analytical result.

In [ ]:
import pinns
import torch
import numpy as np
import matplotlib.pyplot as plt

## 1. Define Parameters

The thermal diffusivity $\alpha$ controls how fast heat spreads. A larger value means faster diffusion.

In [ ]:
# Physical parameter
alpha = 0.01  # Thermal diffusivity

# Domain bounds
x_min, x_max = 0.0, 1.0
t_min, t_max = 0.0, 1.0

## 2. Define the Domain

We create a 2D domain for $(x, t)$ and use Latin Hypercube Sampling for better coverage of the domain.

In [ ]:
domain = pinns.DomainCubic(
    xmin=[x_min, t_min],
    xmax=[x_max, t_max],
    sampling_method="latin_hypercube"
)

print(f"Domain dimensions: {domain.n_dims}")
print(f"Domain bounds: x ∈ [{x_min}, {x_max}], t ∈ [{t_min}, {t_max}]")

## 3. Add Boundary Conditions

We have three boundary conditions:
1. **Left boundary**: $u(0, t) = 0$
2. **Right boundary**: $u(1, t) = 0$
3. **Initial condition**: $u(x, 0) = \sin(\pi x)$

In [ ]:
# Left boundary: u(0, t) = 0
domain.add_dirichlet(
    boundary=(0, None),  # x = x_min, all t
    value=0.0,
    component=0,
    name="left"
)

# Right boundary: u(1, t) = 0
domain.add_dirichlet(
    boundary=(1, None),  # x = x_max, all t
    value=0.0,
    component=0,
    name="right"
)

# Initial condition: u(x, 0) = sin(pi * x)
domain.add_dirichlet(
    boundary=(None, 0),  # all x, t = t_min
    value=lambda x: torch.sin(np.pi * x[:, 0:1]),
    component=0,
    name="initial"
)

print(f"Number of boundary conditions: {len(domain.boundary_conditions)}")

## 4. Define the PDE Residual

The PDE residual is the equation that should equal zero when the PDE is satisfied:

$$\text{residual} = \frac{\partial u}{\partial t} - \alpha \frac{\partial^2 u}{\partial x^2} = 0$$

In [ ]:
def heat_equation(X, V, params):
    """
    Heat equation residual.
    
    Args:
        X: Input tensor (batch, 2) with columns [x, t]
        V: Network output (batch, 1) with column [u]
        params: Dictionary with 'fixed', 'infer', 'internal' keys
    
    Returns:
        Residual tensor (should be zero when PDE is satisfied)
    """
    alpha = params["fixed"]["alpha"]
    
    # Compute derivatives using automatic differentiation
    u_t = pinns.derivative(V, X, component=0, order=(1,))    # ∂u/∂t
    u_xx = pinns.derivative(V, X, component=0, order=(0, 0)) # ∂²u/∂x²
    
    # Heat equation: u_t = alpha * u_xx
    return u_t - alpha * u_xx

## 5. Define the Analytical Solution (for comparison)

Since we know the analytical solution, we can provide it to the trainer for error visualization.

In [ ]:
def analytical_solution(X, params):
    """
    Analytical solution: u(x, t) = exp(-alpha * pi^2 * t) * sin(pi * x)
    """
    alpha = params["fixed"]["alpha"]
    x = X[:, 0:1]
    t = X[:, 1:2]
    
    return np.exp(-alpha * np.pi**2 * t) * np.sin(np.pi * x)

## 6. Create the Problem

The `Problem` class combines the domain, PDE, and parameters into a single object.

In [ ]:
problem = pinns.Problem(
    domain=domain,
    pde_fn=heat_equation,
    input_names=["x", "t"],
    output_names=["u"],
    output_range=(0, 1),  # Expected output range for normalization
    params={"alpha": alpha},
    solution=analytical_solution  # For error tracking
)

print(f"Problem: {problem.n_dims}D input → {problem.n_outputs}D output")

## 7. Create the Neural Network

We use a fully-connected network with 3 hidden layers of 64 neurons each.

The `tanh` activation is commonly used in PINNs as it provides smooth derivatives.

In [ ]:
network = pinns.FNN(
    layer_sizes=[2, 64, 64, 64, 1],  # 2 inputs → 64 → 64 → 64 → 1 output
    activation="tanh",
    normalize_input=True,
    unnormalize_output=True
)

# Count parameters
n_params = sum(p.numel() for p in network.parameters())
print(f"Network architecture: {network.layer_sizes}")
print(f"Total parameters: {n_params:,}")

## 8. Create the Trainer

The `Trainer` handles the training loop, sampling, and visualization.

In [ ]:
trainer = pinns.Trainer(problem, network)
print(f"Training on device: {trainer.device}")

## 9. Configure and Train

We use:
- More samples for the PDE (interior) than boundaries
- Higher weights for boundary conditions to enforce them more strongly
- Adam optimizer for initial training

In [ ]:
trainer.compile(
    train_samples={
        "pde": 5000,      # Interior PDE samples
        "left": 200,      # Left boundary samples
        "right": 200,     # Right boundary samples
        "initial": 500    # Initial condition samples
    },
    test_samples={
        "pde": 500,
        "left": 50,
        "right": 50,
        "initial": 100
    },
    weights={
        "pde": 1.0,
        "left": 10.0,     # Higher weight for BCs
        "right": 10.0,
        "initial": 50.0   # Highest weight for IC
    },
    optimizer="adam",
    learning_rate=1e-3,
    epochs=10000,
    print_each=500,
    show_plots=True
)

trainer.train()

## 10. Fine-tune with L-BFGS (Optional)

L-BFGS often provides better convergence for the final refinement phase.

In [ ]:
trainer.compile(
    optimizer="lbfgs",
    epochs=500,
    print_each=50,
    show_plots=True
)

trainer.train()

## 11. Visualize the Solution

Let's compare the PINN solution with the analytical solution.

In [ ]:
# Create evaluation grid
n_x, n_t = 100, 50
x = np.linspace(x_min, x_max, n_x)
t = np.linspace(t_min, t_max, n_t)
X_grid, T_grid = np.meshgrid(x, t)
points = np.column_stack([X_grid.ravel(), T_grid.ravel()])

# PINN prediction
X_tensor = torch.tensor(points, dtype=torch.float32)
with torch.no_grad():
    u_pred = network(X_tensor).numpy()
U_pred = u_pred.reshape(n_t, n_x)

# Analytical solution
params = {"fixed": {"alpha": alpha}, "infer": {}, "internal": {}}
u_exact = analytical_solution(points, params)
U_exact = u_exact.reshape(n_t, n_x)

# Error
U_error = np.abs(U_pred - U_exact)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# PINN solution
im1 = axes[0].pcolormesh(X_grid, T_grid, U_pred, shading='auto', cmap='viridis')
axes[0].set_xlabel('x')
axes[0].set_ylabel('t')
axes[0].set_title('PINN Solution')
plt.colorbar(im1, ax=axes[0], label='u(x,t)')

# Analytical solution
im2 = axes[1].pcolormesh(X_grid, T_grid, U_exact, shading='auto', cmap='viridis')
axes[1].set_xlabel('x')
axes[1].set_ylabel('t')
axes[1].set_title('Analytical Solution')
plt.colorbar(im2, ax=axes[1], label='u(x,t)')

# Error
im3 = axes[2].pcolormesh(X_grid, T_grid, U_error, shading='auto', cmap='hot')
axes[2].set_xlabel('x')
axes[2].set_ylabel('t')
axes[2].set_title(f'Absolute Error (max: {U_error.max():.2e})')
plt.colorbar(im3, ax=axes[2], label='|error|')

plt.tight_layout()
plt.show()

## 12. Check Solution at Specific Times

In [ ]:
# Plot solution profiles at different times
times = [0.0, 0.25, 0.5, 0.75, 1.0]
x_plot = np.linspace(x_min, x_max, 100)

plt.figure(figsize=(10, 6))

for t_val in times:
    # Create points
    points = np.column_stack([x_plot, np.full_like(x_plot, t_val)])
    
    # PINN prediction
    X_tensor = torch.tensor(points, dtype=torch.float32)
    with torch.no_grad():
        u_pred = network(X_tensor).numpy().flatten()
    
    # Analytical solution
    u_exact = np.exp(-alpha * np.pi**2 * t_val) * np.sin(np.pi * x_plot)
    
    # Plot
    plt.plot(x_plot, u_pred, '-', label=f't={t_val} (PINN)', linewidth=2)
    plt.plot(x_plot, u_exact, '--', label=f't={t_val} (exact)', linewidth=1, alpha=0.7)

plt.xlabel('x')
plt.ylabel('u(x, t)')
plt.title('Temperature Profile at Different Times')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Total loss
axes[0].semilogy(trainer.history['epoch'], trainer.history['loss'])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Total Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

# Solution error (if available)
if trainer.history['solution_error']:
    axes[1].semilogy(trainer.history['epoch'], trainer.history['solution_error'])
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Solution Error')
    axes[1].set_title('Error vs Analytical Solution')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final loss: {trainer.history['loss'][-1]:.2e}")
print(f"Final error: {trainer.history['solution_error'][-1]:.2e}" if trainer.history['solution_error'] else "N/A")

## Summary

In this notebook, we:

1. **Defined the problem**: 1D heat equation with Dirichlet boundary conditions
2. **Created a domain**: Using `DomainCubic` with Latin Hypercube sampling
3. **Added boundary conditions**: Left, right, and initial conditions
4. **Defined the PDE residual**: Using `pinns.derivative` for automatic differentiation
5. **Trained a PINN**: Using Adam + L-BFGS optimization
6. **Compared with analytical solution**: Achieving good agreement

### Tips for Improvement

- **Increase network depth/width** for more complex solutions
- **Use FBPINN** for problems with sharp features or multi-scale behavior
- **Adjust weights** to balance PDE and BC losses
- **Increase training samples** for better accuracy